Dependent Variable - %Change Employment

In [1]:
import pandas as pd
from pathlib import Path
import shutil

countries = [
    "Germany", "Netherlands", "Belgium", "Austria", "Denmark",
    "Sweden", "Norway", "Finland", "Japan", "Korea, Rep.",
    "Switzerland", "United States", "United Kingdom", "Canada",
    "Australia", "New Zealand"
]

unemployment_df = pd.read_csv("../data/raw/Dependent Var/Unemployment.csv")

unemployment_df = unemployment_df[
    unemployment_df["Country Name"].isin(countries)
].copy()

unemployment_df = unemployment_df.replace("..", "NA")

years = list(range(2000, 2024))

year_cols = []
for_year_debug = []

for year in years:
    matching_cols = [
        col for col in unemployment_df.columns
        if str(year) in str(col)
    ]
    if matching_cols:
        year_cols.append(matching_cols[0])
    else:
        for_year_debug.append(year)

columns_to_keep = ["Country Name"] + year_cols

unemployment_clean = unemployment_df[columns_to_keep].copy()
unemployment_clean = unemployment_clean.rename(columns={"Country Name": "Country"})

output_path = Path("../data/clean/Unemployment_clean.csv")
fallback_path = Path("../data/clean/Unemployment_clean_new.csv")

try:
    unemployment_clean.to_csv(output_path, index=False)
    print(f"Cleaned unemployment data saved to {output_path}")
except PermissionError:
    unemployment_clean.to_csv(fallback_path, index=False)
    print(f"Original file is open/locked. Saved instead to {fallback_path}")

print("Years missing from raw file:", for_year_debug)
display(unemployment_clean.head())

Cleaned unemployment data saved to ..\data\clean\Unemployment_clean.csv
Years missing from raw file: []


,Country,2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],2008 [YR2008],...,2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023]
0,New Zealand,6.130,5.433,5.282,4.745,4.005,3.807,3.857,3.661,4.167,...,5.428,5.415,5.147,4.736,4.335,4.109,4.595,3.776,3.299,3.734
1,Germany,7.917,7.773,8.482,9.779,10.727,11.193,10.277,8.732,7.508,...,4.979,4.612,4.104,3.781,3.384,3.163,3.881,3.594,3.137,3.071
2,Netherlands,2.725,2.119,2.554,3.593,4.646,5.872,5.007,4.153,3.658,...,7.419,6.872,6.007,4.837,3.832,3.379,3.820,4.209,3.526,3.535
3,Belgium,6.586,6.178,6.910,7.680,7.363,8.440,8.246,7.459,6.976,...,8.523,8.482,7.830,7.090,5.941,5.364,5.545,6.248,5.570,5.547
4,Austria,4.687,4.007,4.852,4.779,5.969,5.682,5.320,4.909,4.198,...,5.674,5.802,6.064,5.561,4.933,4.560,5.201,6.459,4.992,5.264


Independent Variable - Labour Market Coordination (COORD)

In [2]:
# Load and clean COORD data
countries = [
    'Germany', 'Netherlands', 'Belgium', 'Austria', 'Denmark', 'Sweden', 'Norway', 'Finland',
    'Japan', 'Korea, Rep.', 'Switzerland', 'United States', 'United Kingdom', 'Canada', 'Australia', 'New Zealand'
]

coord_df = pd.read_csv(Path('../data/raw/Independent Var/COORD.csv'), encoding='latin1', skiprows=5)

# Normalize headers and required columns
coord_df.columns = [col.strip() for col in coord_df.columns]
if 'Country' not in coord_df.columns or 'Year' not in coord_df.columns or 'Coord' not in coord_df.columns:
    raise ValueError('COORD file is missing Country, Year, or Coord columns')

coord_df['Year'] = pd.to_numeric(coord_df['Year'], errors='coerce')

# Filter for selected countries and target years
coord_df = coord_df[coord_df['Country'].isin(countries) & coord_df['Year'].between(2000, 2023)]

# Replace empty or whitespace-only values with NA
coord_df = coord_df.replace(r'^\s*$', 'NA', regex=True).fillna('NA')

# Keep only Country, Year, and Coord columns
coord_df = coord_df[['Country', 'Year', 'Coord']]

# Save the cleaned data safely
output_path = Path('../data/clean/COORD_clean.csv')
temp_path = output_path.with_suffix('.csv.tmp')
coord_df.to_csv(temp_path, index=False)

try:
    temp_path.replace(output_path)
    print(f"Cleaned COORD data saved to {output_path}")
except PermissionError:
    fallback_path = output_path.with_name('COORD_clean_fallback.csv')
    temp_path.replace(fallback_path)
    print(f"Could not replace existing file; saved cleaned data to {fallback_path}")

Cleaned COORD data saved to ..\data\clean\COORD_clean.csv


Independent Variable - Labour Market Coordination (COORD) - Imputed Assumption

Since COORD is relatively time-invariant, for years with NA, it was replaced with the data from the year before it, if not, after it.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

# -----------------------------
# Clean and Impute COORD Data
# -----------------------------

countries = [
    "Germany", "Netherlands", "Belgium", "Austria", "Denmark",
    "Sweden", "Norway", "Finland", "Japan", "Korea, Rep.",
    "Switzerland", "United States", "United Kingdom", "Canada",
    "Australia", "New Zealand"
]

years = list(range(2000, 2024))

# File paths
raw_path = Path("../data/raw/Independent Var/COORD.csv")
clean_dir = Path("../data/clean")
clean_dir.mkdir(parents=True, exist_ok=True)

output_path = clean_dir / "COORD_clean_imputed.csv"

# -----------------------------
# Load raw COORD data
# -----------------------------

coord_df = pd.read_csv(
    raw_path,
    encoding="latin1",
    skiprows=5
)

# Standardise column names
coord_df.columns = coord_df.columns.astype(str).str.strip()

print("Columns found:")
print(coord_df.columns.tolist())

# -----------------------------
# Rename likely columns
# -----------------------------

rename_map = {
    "Country": "Country",
    "country": "Country",
    "Country Name": "Country",
    "Year": "Year",
    "year": "Year",
    "coord": "Coord",
    "COORD": "Coord",
    "Coord": "Coord"
}

coord_df = coord_df.rename(columns={k: v for k, v in rename_map.items() if k in coord_df.columns})

# Check required columns
required_cols = ["Country", "Year", "Coord"]
missing_cols = [col for col in required_cols if col not in coord_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}. Please show me the column names printed above.")

# -----------------------------
# Clean values
# -----------------------------

coord_df["Country"] = coord_df["Country"].astype(str).str.strip()
coord_df["Year"] = pd.to_numeric(coord_df["Year"], errors="coerce")
coord_df["Coord"] = pd.to_numeric(coord_df["Coord"], errors="coerce")

# Rename South Korea if needed
coord_df["Country"] = coord_df["Country"].replace({
    "Korea, Republic of": "Korea, Rep.",
    "South Korea": "Korea, Rep."
})

# Keep selected countries and years
coord_df = coord_df[
    (coord_df["Country"].isin(countries)) &
    (coord_df["Year"].isin(years))
].copy()

coord_df = coord_df[["Country", "Year", "Coord"]]

# -----------------------------
# Create full country-year panel
# -----------------------------

full_panel = pd.MultiIndex.from_product(
    [countries, years],
    names=["Country", "Year"]
).to_frame(index=False)

coord_clean = full_panel.merge(
    coord_df,
    on=["Country", "Year"],
    how="left"
)

# -----------------------------
# Impute missing COORD
# -----------------------------

coord_clean = coord_clean.sort_values(["Country", "Year"])

# First use previous year within country, then next available year
coord_clean["Coord_Imputed"] = (
    coord_clean
    .groupby("Country")["Coord"]
    .transform(lambda x: x.ffill().bfill())
)

# Track which values were imputed
coord_clean["Coord_Was_Imputed"] = coord_clean["Coord"].isna() & coord_clean["Coord_Imputed"].notna()

# Final cleaned COORD variable
coord_clean["Coord"] = coord_clean["Coord_Imputed"]

coord_clean = coord_clean.drop(columns=["Coord_Imputed"])

# -----------------------------
# Save CSV safely
# -----------------------------

temp_path = output_path.with_suffix(".tmp.csv")
coord_clean.to_csv(temp_path, index=False, na_rep="NA")

try:
    temp_path.replace(output_path)
except PermissionError:
    fallback_path = clean_dir / "COORD_clean_imputed_fallback.csv"
    shutil.copy(temp_path, fallback_path)
    print(f"Permission error. File saved instead to: {fallback_path}")

print(f"Clean imputed COORD data saved to: {output_path}")
print(coord_clean.head(30))
print("\nMissing values remaining:")
print(coord_clean.isna().sum())

Columns found:
['Country', 'ISO', 'Year', 'ED', 'ED_private', 'UD_hist', 'UD_male_hist', 'UD_female_hist', 'UM_female_hist', 'UD_private_hist', 'UD_public_hist', 'UM_public_hist', 'AdjCov_hist', 'AdjCov_private_hist', 'AdjCov_public_hist', 'Level', 'Multilevel', 'rAEB', 'FAV', 'OCG', 'OCT', 'Ext', 'Coord', 'Type', 'SPA_Sign', 'TC', 'WC', 'WC_negot']
Clean imputed COORD data saved to: ..\data\clean\COORD_clean_imputed.csv
       Country  Year  Coord  Coord_Was_Imputed
336  Australia  2000    2.0              False
337  Australia  2001    2.0              False
338  Australia  2002    2.0              False
339  Australia  2003    2.0              False
340  Australia  2004    2.0              False
341  Australia  2005    1.0              False
342  Australia  2006    1.0              False
343  Australia  2007    1.0              False
344  Australia  2008    1.0              False
345  Australia  2009    2.0              False
346  Australia  2010    2.0              False
347  Austra

Independent Variable - For Robustness Checks - Union Density and Bargaining Coverage

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# Clean Union Density and Bargaining Coverage Data
# Source: ICTWSS / OECD-AIAS
# --------------------------------------------------

countries = [
    "Germany", "Netherlands", "Belgium", "Austria", "Denmark",
    "Sweden", "Norway", "Finland", "Japan", "Korea",
    "Switzerland", "United States", "United Kingdom", "Canada",
    "Australia", "New Zealand"
]

country_fix = {
    "Korea": "South Korea",
    "Korea, Rep.": "South Korea",
    "USA": "United States",
    "UK": "United Kingdom"
}

years = list(range(2000, 2024))

raw_path = Path("../data/raw/Independent Var/COORD.csv")
clean_path = Path("../data/clean/InstitutionalVariables_clean.csv")
clean_path.parent.mkdir(parents=True, exist_ok=True)

# Read ICTWSS file
ictwss = pd.read_csv(
    raw_path,
    skiprows=5,
    encoding="latin1"
)

# Clean column names
ictwss.columns = ictwss.columns.astype(str).str.strip()

print("Columns found:")
print(ictwss.columns.tolist())

# Keep relevant columns
needed_cols = ["Country", "Year", "UD_hist", "AdjCov_hist", "Coord"]

missing = [col for col in needed_cols if col not in ictwss.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Check printed column list above.")

institutional_clean = ictwss[needed_cols].copy()

# Rename variables
institutional_clean = institutional_clean.rename(columns={
    "UD_hist": "UnionDensity",
    "AdjCov_hist": "BargainingCoverage",
    "Coord": "COORD"
})

# Clean country names
institutional_clean["Country"] = institutional_clean["Country"].astype(str).str.strip()
institutional_clean["Country"] = institutional_clean["Country"].replace(country_fix)

# Clean year
institutional_clean["Year"] = pd.to_numeric(
    institutional_clean["Year"],
    errors="coerce"
)

# Convert variables to numeric
for col in ["UnionDensity", "BargainingCoverage", "COORD"]:
    institutional_clean[col] = pd.to_numeric(
        institutional_clean[col],
        errors="coerce"
    )

# Replace ICTWSS missing codes with NA
institutional_clean = institutional_clean.replace({
    -99: np.nan,
    -88: np.nan,
    -77: np.nan
})

# Final country names
final_countries = [
    "Germany", "Netherlands", "Belgium", "Austria", "Denmark",
    "Sweden", "Norway", "Finland", "Japan", "South Korea",
    "Switzerland", "United States", "United Kingdom", "Canada",
    "Australia", "New Zealand"
]

# Filter sample
institutional_clean = institutional_clean[
    institutional_clean["Country"].isin(final_countries) &
    institutional_clean["Year"].isin(years)
].copy()

# Create full country-year panel
full_panel = pd.MultiIndex.from_product(
    [final_countries, years],
    names=["Country", "Year"]
).to_frame(index=False)

institutional_clean = full_panel.merge(
    institutional_clean,
    on=["Country", "Year"],
    how="left"
)

# Impute slow-moving institutional variables using adjacent years
for col in ["UnionDensity", "BargainingCoverage", "COORD"]:
    institutional_clean[col + "_Was_Imputed"] = institutional_clean[col].isna()
    
    institutional_clean[col] = (
        institutional_clean
        .groupby("Country")[col]
        .transform(lambda x: x.ffill().bfill())
    )
    
    institutional_clean[col + "_Was_Imputed"] = (
        institutional_clean[col + "_Was_Imputed"] &
        institutional_clean[col].notna()
    )

# Save clean file
institutional_clean.to_csv(clean_path, index=False, na_rep="NA")

print(f"Clean institutional variables saved to: {clean_path}")
print("\nMissing values remaining:")
print(institutional_clean.isna().sum())

display(institutional_clean.head(30))


Columns found:
['Country', 'ISO', 'Year', 'ED', 'ED_private', 'UD_hist', 'UD_male_hist', 'UD_female_hist', 'UM_female_hist', 'UD_private_hist', 'UD_public_hist', 'UM_public_hist', 'AdjCov_hist', 'AdjCov_private_hist', 'AdjCov_public_hist', 'Level', 'Multilevel', 'rAEB', 'FAV', 'OCG', 'OCT', 'Ext', 'Coord', 'Type', 'SPA_Sign', 'TC', 'WC', 'WC_negot']
Clean institutional variables saved to: ..\data\clean\InstitutionalVariables_clean.csv

Missing values remaining:
Country                            0
Year                               0
UnionDensity                      24
BargainingCoverage                24
COORD                             24
UnionDensity_Was_Imputed           0
BargainingCoverage_Was_Imputed     0
COORD_Was_Imputed                  0
dtype: int64


,Country,Year,UnionDensity,BargainingCoverage,COORD,UnionDensity_Was_Imputed,BargainingCoverage_Was_Imputed,COORD_Was_Imputed
0,Germany,2000,22.0,67.8,4.0,False,False,False
1,Germany,2001,21.4,68.8,4.0,False,False,False
2,Germany,2002,21.1,67.8,4.0,False,False,False
3,Germany,2003,20.7,67.6,4.0,False,False,False
4,Germany,2004,19.8,65.8,4.0,False,False,False
5,Germany,2005,19.6,64.9,4.0,False,False,False
6,Germany,2006,19.1,63.3,4.0,False,False,False
7,Germany,2007,18.4,61.7,4.0,False,False,False
8,Germany,2008,17.8,61.3,4.0,False,False,False
9,Germany,2009,17.6,61.7,4.0,False,False,False


Control Variable - Export %(dependency)

In [4]:
# Load and clean export dependency data
countries = [
    'Germany', 'Netherlands', 'Belgium', 'Austria', 'Denmark', 'Sweden', 'Norway', 'Finland',
    'Japan', 'Korea, Rep.', 'Switzerland', 'United States', 'United Kingdom', 'Canada', 'Australia', 'New Zealand'
]

expdep_df = pd.read_csv(Path('../data/raw/Control Var/ExpDep.csv'))

# Filter for selected countries
expdep_df = expdep_df[expdep_df['Country Name'].isin(countries)]

# Replace empty values and missing values with NA
expdep_df = expdep_df.replace(['', ' '], 'NA').fillna('NA')

# Keep Country Name plus years 2000-2023 where available
years = [f'{year} [YR{year}]' for year in range(2000, 2024)]
columns_to_keep = ['Country Name'] + [year for year in years if year in expdep_df.columns]
expdep_clean = expdep_df[columns_to_keep]

# Save the cleaned data
output_path = Path('../data/clean/ExpDep_clean.csv')
expdep_clean.to_csv(output_path, index=False)

print(f"Cleaned export dependency data saved to {output_path}")

Cleaned export dependency data saved to ..\data\clean\ExpDep_clean.csv


Control Variable - GDP

In [5]:
# Load and clean GDP data
countries = [
    'Germany', 'Netherlands', 'Belgium', 'Austria', 'Denmark', 'Sweden', 'Norway', 'Finland',
    'Japan', 'Korea, Rep.', 'Switzerland', 'United States', 'United Kingdom', 'Canada', 'Australia', 'New Zealand'
]

gdp_df = pd.read_csv(Path('../data/raw/Control Var/GDP.csv'))

# Filter for selected countries
gdp_df = gdp_df[gdp_df['Country Name'].isin(countries)]

# Replace empty values and missing values with NA
gdp_df = gdp_df.replace(['', ' '], 'NA').fillna('NA')

# Keep Country Name plus years 2000-2023 where available
years = [f'{year} [YR{year}]' for year in range(2000, 2024)]
columns_to_keep = ['Country Name'] + [year for year in years if year in gdp_df.columns]
gdp_clean = gdp_df[columns_to_keep]

# Save the cleaned data
output_path = Path('../data/clean/GDP_clean.csv')
gdp_clean.to_csv(output_path, index=False)

print(f"Cleaned GDP data saved to {output_path}")

Cleaned GDP data saved to ..\data\clean\GDP_clean.csv


Control Variable - %Inflation

In [6]:
# Load and clean inflation data
countries = [
    'Germany', 'Netherlands', 'Belgium', 'Austria', 'Denmark', 'Sweden', 'Norway', 'Finland',
    'Japan', 'Korea, Rep.', 'Switzerland', 'United States', 'United Kingdom', 'Canada', 'Australia', 'New Zealand'
]

inflation_df = pd.read_csv(Path('../data/raw/Control Var/Inflation.csv'), encoding='latin1')

# Filter for selected countries
inflation_df = inflation_df[inflation_df['Country Name'].isin(countries)]

# Replace empty or whitespace-only values with NA
inflation_df = inflation_df.replace(r'^\s*$', 'NA', regex=True).fillna('NA')

# Keep Country Name plus years 2000-2023 where available
years = [f'{year} [YR{year}]' for year in range(2000, 2024)]
columns_to_keep = ['Country Name'] + [year for year in years if year in inflation_df.columns]
inflation_clean = inflation_df[columns_to_keep]

# Save the cleaned data
output_path = Path('../data/clean/Inflation_clean.csv')
inflation_clean.to_csv(output_path, index=False)

print(f"Cleaned inflation data saved to {output_path}")

Cleaned inflation data saved to ..\data\clean\Inflation_clean.csv


Control Variable - %Gov Spending

In [7]:
# Load and clean government spending data
countries = [
    'Germany', 'Netherlands', 'Belgium', 'Austria', 'Denmark', 'Sweden', 'Norway', 'Finland',
    'Japan', 'Korea, Rep.', 'Switzerland', 'United States', 'United Kingdom', 'Canada', 'Australia', 'New Zealand'
]

govspend_df = pd.read_csv(Path('../data/raw/Control Var/GovSpend.csv'))

# Filter for selected countries
govspend_df = govspend_df[govspend_df['Country Name'].isin(countries)]

# Replace empty or missing values with NA
govspend_df = govspend_df.replace(r'^\s*$', 'NA', regex=True).fillna('NA')

# Keep Country Name plus years 2000-2023 where available
years = [f'{year} [YR{year}]' for year in range(2000, 2024)]
columns_to_keep = ['Country Name'] + [year for year in years if year in govspend_df.columns]
govspend_clean = govspend_df[columns_to_keep]

# Save the cleaned data
output_path = Path('../data/clean/GovSpend_clean.csv')
govspend_clean.to_csv(output_path, index=False)

print(f"Cleaned government spending data saved to {output_path}")

Cleaned government spending data saved to ..\data\clean\GovSpend_clean.csv


Final Panel - A combination CSV of all cleaned data

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# --------------------------------------------------
# Create Final Merged Panel Dataset
# --------------------------------------------------

clean_dir = Path("../data/clean")
output_path = clean_dir / "final_panel.csv"

countries = [
    "Germany", "Netherlands", "Belgium", "Austria", "Denmark",
    "Sweden", "Norway", "Finland", "Japan", "South Korea",
    "Switzerland", "United States", "United Kingdom", "Canada",
    "Australia", "New Zealand"
]

years = list(range(2000, 2024))

# --------------------------------------------------
# Helper function: works for wide or long format
# --------------------------------------------------

def load_clean_file(filename, value_col, final_col=None):
    path = clean_dir / filename
    
    if not path.exists():
        raise FileNotFoundError(f"Could not find: {path}")
    
    df = pd.read_csv(path)
    df.columns = df.columns.astype(str).str.strip()
    
    rename_map = {
        "Country Name": "Country",
        "country": "Country",
        "Countries": "Country",
        "Year ": "Year",
        "year": "Year",
        "Time": "Year"
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    
    if "Country" not in df.columns:
        raise ValueError(f"{filename} needs a Country column. Current columns: {df.columns.tolist()}")
    
    # Long format
    if "Year" in df.columns:
        if value_col not in df.columns:
            possible_cols = [
                c for c in df.columns 
                if c not in ["Country", "Year", "Coord_Was_Imputed"]
            ]
            if len(possible_cols) == 1:
                df = df.rename(columns={possible_cols[0]: value_col})
            else:
                raise ValueError(
                    f"Could not identify {value_col} in {filename}. "
                    f"Columns: {df.columns.tolist()}"
                )
        
        df = df[["Country", "Year", value_col]].copy()
    
    # Wide format
    else:
        year_cols = []
        for col in df.columns:
            match = re.search(r"(19|20)\d{2}", col)
            if match:
                year = int(match.group())
                if year in years:
                    year_cols.append(col)
        
        if len(year_cols) == 0:
            raise ValueError(
                f"No year columns found in {filename}. "
                f"Columns: {df.columns.tolist()}"
            )
        
        df = df[["Country"] + year_cols].copy()
        
        df = df.melt(
            id_vars="Country",
            value_vars=year_cols,
            var_name="Year",
            value_name=value_col
        )
        
        df["Year"] = df["Year"].astype(str).str.extract(r"((?:19|20)\d{2})").astype(int)
    
    # Clean values
    df["Country"] = df["Country"].astype(str).str.strip()
    df["Country"] = df["Country"].replace({
        "Korea, Rep.": "South Korea",
        "Korea, Republic of": "South Korea",
        "USA": "United States",
        "UK": "United Kingdom"
    })
    
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    
    df = df[
        (df["Country"].isin(countries)) &
        (df["Year"].isin(years))
    ].copy()
    
    if final_col is not None:
        df = df.rename(columns={value_col: final_col})
        value_col = final_col
    
    return df[["Country", "Year", value_col]]

# --------------------------------------------------
# Load cleaned files using your actual filenames
# --------------------------------------------------

unemployment = load_clean_file("Unemployment_clean.csv", "Unemployment")
coord = load_clean_file("COORD_clean_imputed.csv", "Coord", final_col="COORD")
gdp = load_clean_file("GDP_clean.csv", "GDPGrowth")
inflation = load_clean_file("Inflation_clean.csv", "Inflation")
export_dep = load_clean_file("ExpDep_clean.csv", "ExportDependence")
gov_spend = load_clean_file("GovSpend_clean.csv", "GovSpend")

# --------------------------------------------------
# Create balanced country-year panel
# --------------------------------------------------

panel = pd.MultiIndex.from_product(
    [countries, years],
    names=["Country", "Year"]
).to_frame(index=False)

# --------------------------------------------------
# Merge all datasets
# --------------------------------------------------

for df in [unemployment, coord, gdp, inflation, export_dep, gov_spend]:
    panel = panel.merge(df, on=["Country", "Year"], how="left")

# --------------------------------------------------
# Create resilience variable
# --------------------------------------------------

panel = panel.sort_values(["Country", "Year"]).reset_index(drop=True)

panel["Delta_Unemployment"] = (
    panel.groupby("Country")["Unemployment"].diff()
)

# --------------------------------------------------
# Reorder columns
# --------------------------------------------------

panel = panel[
    [
        "Country",
        "Year",
        "Unemployment",
        "Delta_Unemployment",
        "COORD",
        "GDPGrowth",
        "Inflation",
        "ExportDependence",
        "GovSpend"
    ]
]

# --------------------------------------------------
# Save final dataset
# --------------------------------------------------

panel.to_csv(output_path, index=False, na_rep="NA")

print(f"Final panel dataset saved to: {output_path}")
print("Shape:", panel.shape)

print("\nMissing values by column:")
print(panel.isna().sum())

print("\nPreview:")
display(panel.head(30))

Final panel dataset saved to: ..\data\clean\final_panel.csv
Shape: (384, 9)

Missing values by column:
Country                0
Year                   0
Unemployment           0
Delta_Unemployment    16
COORD                  0
GDPGrowth              0
Inflation              0
ExportDependence       0
GovSpend               3
dtype: int64

Preview:


,Country,Year,Unemployment,Delta_Unemployment,COORD,GDPGrowth,Inflation,ExportDependence,GovSpend
0,Australia,2000,6.288,NaN,2.0,3.916057,4.457437,19.356083,18.505430
1,Australia,2001,6.747,0.459,2.0,2.031459,4.407130,22.109757,18.507007
2,Australia,2002,6.375,-0.372,2.0,3.949805,2.981571,20.692768,18.318430
3,Australia,2003,5.933,-0.442,2.0,3.091280,2.732598,19.025563,18.380148
4,Australia,2004,5.399,-0.534,2.0,4.218941,2.343257,17.135063,18.273056
5,Australia,2005,5.036,-0.363,1.0,3.155481,2.691829,18.195117,18.352087
6,Australia,2006,4.785,-0.251,1.0,2.770926,3.555287,19.819356,18.275777
7,Australia,2007,4.381,-0.404,1.0,3.789471,2.327614,20.145895,18.138391
8,Australia,2008,4.242,-0.139,1.0,3.623985,4.350297,20.104412,18.053979
9,Australia,2009,5.565,1.323,2.0,1.954848,1.771120,22.936907,18.371414
